In [ ]:
# ============================================================
# Geospatial Statistics Mini Project (GNR 640)
# ============================================================

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import pearsonr, ks_2samp
from pykrige.ok import OrdinaryKriging

# ------------------------------------------------------------
# 1. DEFINE GRID (2° x 2°)
# ------------------------------------------------------------

lat_grid = np.arange(25, 50, 2)
lon_grid = np.arange(-125, -65, 2)
lon_mesh, lat_mesh = np.meshgrid(lon_grid, lat_grid)

# ------------------------------------------------------------
# 2. VARIABLES
# ------------------------------------------------------------

variables = [
    'temperature',
    'precipitation',
    'humidity',
    'soil_moisture',
    'soil_temperature'
]

# ------------------------------------------------------------
# 3. LOAD DATA (PLACEHOLDER PATHS)
# ------------------------------------------------------------

station_df = pd.read_csv("/content/drive/MyDrive/GNR640/All_CONUS_Station_Information_V2.xlsx")
era5_ds = xr.open_dataset("/content/era5_land_usa_data")

station_df['date'] = pd.to_datetime(station_df['date'])

# ------------------------------------------------------------
# 4. PREPROCESSING
# ------------------------------------------------------------

station_df = station_df.interpolate(method='linear')

station_df = station_df[
    (station_df['date'] >= '2006-01-01') &
    (station_df['date'] <= '2021-12-31')
]

dates = pd.date_range("2006-01-01", "2021-12-31", freq="D")

# ------------------------------------------------------------
# 5. INTERPOLATION FUNCTIONS
# ------------------------------------------------------------

def idw_interpolation(x, y, z, xi, yi, power=2):
    tree = cKDTree(np.c_[x, y])
    dist, idx = tree.query(np.c_[xi.flatten(), yi.flatten()], k=5)

    weights = 1 / (dist**power + 1e-12)
    weights /= weights.sum(axis=1, keepdims=True)

    zi = np.sum(weights * z[idx], axis=1)
    return zi.reshape(xi.shape)


def kriging_interpolation(x, y, z, xi, yi):
    OK = OrdinaryKriging(
        x, y, z,
        variogram_model='spherical',
        verbose=False,
        enable_plotting=False
    )
    zi, ss = OK.execute('grid', xi[0], yi[:, 0])
    return zi


def nearest_neighbor(x, y, z, xi, yi):
    tree = cKDTree(np.c_[x, y])
    _, idx = tree.query(np.c_[xi.flatten(), yi.flatten()])
    return z[idx].reshape(xi.shape)

# ------------------------------------------------------------
# 6. MAIN LOOP (ALL VARIABLES)
# ------------------------------------------------------------

all_results = []

for variable in variables:

    print(f"Processing variable: {variable}")
    results = []

    for date in dates[:50]:  # subset for demo (important note)

        daily_data = station_df[station_df['date'] == date]

        if len(daily_data) < 5:
            continue

        x = daily_data['lon'].values
        y = daily_data['lat'].values
        z = daily_data[variable].values

        # --- Interpolation ---
        idw_grid = idw_interpolation(x, y, z, lon_mesh, lat_mesh)
        krig_grid = kriging_interpolation(x, y, z, lon_mesh, lat_mesh)
        nn_grid = nearest_neighbor(x, y, z, lon_mesh, lat_mesh)

        # --- ERA5 Extraction ---
        era5_slice = era5_ds.sel(time=date, method='nearest')

        if variable == 'precipitation':
            era5_resampled = era5_slice[variable].coarsen(
                lat=8, lon=8, boundary='trim').sum()
        else:
            era5_resampled = era5_slice[variable].coarsen(
                lat=8, lon=8, boundary='trim').mean()

        ref = era5_resampled.values.flatten()

        # --- Metrics ---
        for method, grid in zip(['IDW', 'Kriging', 'NN'],
                               [idw_grid, krig_grid, nn_grid]):

            pred = grid.flatten()

            rmse = np.sqrt(np.mean((pred - ref)**2))
            r, _ = pearsonr(pred, ref)

            results.append([date, variable, method, rmse, r])

    df = pd.DataFrame(results,
                      columns=['Date', 'Variable', 'Method', 'RMSE', 'R'])

    all_results.append(df)

# Combine all variables
final_results = pd.concat(all_results)

# ------------------------------------------------------------
# 7. SUMMARY TABLE
# ------------------------------------------------------------

summary_table = final_results.groupby(['Variable', 'Method']).agg({
    'RMSE': 'mean',
    'R': 'mean'
}).reset_index()

print("\n=== FINAL PERFORMANCE TABLE ===")
print(summary_table)

# ------------------------------------------------------------
# 8. KS TEST (Example)
# ------------------------------------------------------------

ks_stat, p_val = ks_2samp(
    final_results[final_results['Method'] == 'Kriging']['RMSE'],
    final_results[final_results['Method'] == 'IDW']['RMSE']
)

print(f"\nKS Statistic: {ks_stat:.3f}, p-value: {p_val:.3f}")

# ------------------------------------------------------------
# 9. VISUALIZATION
# ------------------------------------------------------------

# ---- RMSE Comparison ----
plt.figure(figsize=(8, 5))
for var in variables:
    subset = summary_table[summary_table['Variable'] == var]
    plt.bar(subset['Method'], subset['RMSE'], alpha=0.6, label=var)

plt.title("RMSE Comparison Across Variables")
plt.ylabel("RMSE")
plt.legend()
plt.show()

# ---- Correlation Comparison ----
plt.figure(figsize=(8, 5))
for var in variables:
    subset = summary_table[summary_table['Variable'] == var]
    plt.plot(subset['Method'], subset['R'], marker='o', label=var)

plt.title("Correlation Comparison")
plt.ylabel("Correlation (R)")
plt.legend()
plt.show()

# ---- Time Series ----
ts = final_results.groupby('Date')['RMSE'].mean()

plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values)
plt.title("RMSE Time Series (2006–2021)")
plt.ylabel("RMSE")
plt.xlabel("Time")
plt.show()

# ---- Boxplot ----
plt.figure(figsize=(8, 5))
final_results.boxplot(column='RMSE', by='Method')
plt.title("RMSE Distribution by Method")
plt.suptitle("")
plt.show()

# ------------------------------------------------------------
# 10. SAVE RESULTS
# ------------------------------------------------------------

final_results.to_csv("outputs/final_results.csv", index=False)
summary_table.to_csv("outputs/summary_table.csv", index=False)

# ============================================================
# END OF SCRIPT
# ============================================================